In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
from datetime import datetime

In [2]:
matplotlib.use('tkagg')
%matplotlib auto
%matplotlib auto

Using matplotlib backend: tkagg
Using matplotlib backend: tkagg


In [14]:
# 74执行器绕组过温数据
# 74[0-9]: 101-110列    54: 41-50列
data_root = '/home/fourier/GRX_humanoid/real data/'
data_path = data_root + '251112_GR301AA0017,腿部执行器54，55，74执行器过热/2025-11-12-16-34.csv'

df = pd.read_csv(data_path)
print(df.shape)

/tmp/ipykernel_245826/3425770589.py:6: DtypeWarning: Columns (50,60,110) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


(139391, 121)


In [15]:
start_id = 0
time_strings = df.iloc[start_id:, 0]
time = [datetime.strptime(t[11:], '%H:%M:%S:%f') for t in time_strings]
time_seconds = [(t - time[0]).total_seconds() for t in time]
print(time_seconds[:10])
joint_data = df.iloc[start_id:,:].values

# print(joint_data_54.shape)
# print(joint_data_54[0, :])
# print(joint_data_74.shape)
# print(joint_data_74[0, :])
# print(time_strings.shape)

[0.0, 0.015, 0.03, 0.045, 0.06, 0.075, 0.09, 0.105, 0.12, 0.135]


In [17]:
# 整体关节命名顺序，先下肢后上肢
gr3_joint_names = ["left_hip_pitch_joint", "left_hip_roll_joint", "left_hip_yaw_joint", \
                    "left_knee_pitch_joint", "left_ankle_pitch_joint", "left_ankle_roll_joint", \
                    "right_hip_pitch_joint", "right_hip_roll_joint", "right_hip_yaw_joint",  \
                    "right_knee_pitch_joint", "right_ankle_pitch_joint", "right_ankle_roll_joint",\
                    "waist_yaw_joint", "waist_roll_joint","waist_pitch_joint", \
                    "head_yaw_joint", "head_pitch_joint",\
                    "left_shoulder_pitch_joint", "left_shoulder_roll_joint", "left_shoulder_yaw_joint", "left_elbow_pitch_joint", \
                    "left_wrist_yaw_joint", "left_wrist_pitch_joint", "left_wrist_roll_joint", \
                    "right_shoulder_pitch_joint", "right_shoulder_roll_joint", "right_shoulder_yaw_joint", "right_elbow_pitch_joint", \
                    "right_wrist_yaw_joint", "right_wrist_pitch_joint", "right_wrist_roll_joint"]
lower_joint_names = [   "left_hip_pitch_joint", "left_hip_roll_joint", "left_hip_yaw_joint", \
                        "left_knee_pitch_joint", "left_ankle_pitch_joint", "left_ankle_roll_joint", \
                        "right_hip_pitch_joint", "right_hip_roll_joint", "right_hip_yaw_joint",  \
                        "right_knee_pitch_joint", "right_ankle_pitch_joint", "right_ankle_roll_joint",\
                        "waist_yaw_joint", "waist_roll_joint","waist_pitch_joint"]
upper_joint_names = [   "head_yaw_joint", "head_pitch_joint",\
                        "left_shoulder_pitch_joint", "left_shoulder_roll_joint", "left_shoulder_yaw_joint", "left_elbow_pitch_joint", \
                        "left_wrist_yaw_joint", "left_wrist_pitch_joint", "left_wrist_roll_joint", \
                        "right_shoulder_pitch_joint", "right_shoulder_roll_joint", "right_shoulder_yaw_joint", "right_elbow_pitch_joint", \
                        "right_wrist_yaw_joint", "right_wrist_pitch_joint", "right_wrist_roll_joint"]
# error code 不读取
labels = ["pos", "vel", "current", "tor",
         "MOS temp", "armature temp", "Targ pos", "Targ vel", "Targ current", "error code"]

In [18]:
IP = [10, 11, 12, 13, 14, 15, 16,
      30, 31, 32, 33, 34, 35, 36,
      50, 51, 52, 53, 54, 55,
      70, 71, 72, 73, 74, 75,
      90, 91, 92, 93, 95]  # 全身关节在数据中的索引位置
IP_index = {v: 10*i+1 for i, v in enumerate(IP)}  # 全身关节在数据中的索引位置字典
print(IP_index)

IP_upper = [93, 95, 
            10, 11, 12, 13, 14, 15, 16,
            30, 31, 32, 33, 34, 35, 36]  # 上肢关节在数据中的索引位置

IP_lower = [50, 51, 52, 53, 54, 55,
      70, 71, 72, 73, 74, 75,
      90, 91, 92]  # 下肢关节在数据中的索引位置
IP_index_lower = {v: 10*i+1 for i, v in enumerate(IP_lower)}  # 下肢关节在数据中的索引位置字典
# IP_index_lower = {v: 10*i+141 for i, v in enumerate(IP_lower)}  # 下肢关节在数据中的索引位置字典
print(IP_index_lower)

{10: 1, 11: 11, 12: 21, 13: 31, 14: 41, 15: 51, 16: 61, 30: 71, 31: 81, 32: 91, 33: 101, 34: 111, 35: 121, 36: 131, 50: 141, 51: 151, 52: 161, 53: 171, 54: 181, 55: 191, 70: 201, 71: 211, 72: 221, 73: 231, 74: 241, 75: 251, 90: 261, 91: 271, 92: 281, 93: 291, 95: 301}
{50: 1, 51: 11, 52: 21, 53: 31, 54: 41, 55: 51, 70: 61, 71: 71, 72: 81, 73: 91, 74: 101, 75: 111, 90: 121, 91: 131, 92: 141}


In [19]:
# 单关节各数据
joint_plot_IP = [54, 55, 74]  # 要绘制的关节IP
for ip in joint_plot_IP:
    ip_index = IP_index_lower[ip]
    fig, axs = plt.subplots(2, 3, figsize=(20, 15), sharex=True)
    print("plot ", ip, "index:", ip_index)
    fig.suptitle(f'Joint IP: {ip}', fontsize=16)
    for i, ax in enumerate(axs.flatten()):
        if i < 3:
            ax.plot(time_seconds, joint_data[:, ip_index + i + 6], label=labels[i+6], linestyle='--')
            ax.plot(time_seconds, joint_data[:, ip_index + i], label=labels[i])
        else:
            ax.plot(time_seconds, joint_data[:, ip_index + i], label=labels[i])
        # ax.set_title(labels[i])
        ax.set_xlabel('Time (s)')
        ax.legend()
        ax.grid()
    plt.tight_layout()
    plt.show()

plot  54 index: 41
plot  55 index: 51
plot  74 index: 101


In [ ]:
# for csv only have lower joint data
joint_data_50 = df.iloc[:, 1:10].values
joint_data_51 = df.iloc[:, 11:20].values
joint_data_52 = df.iloc[:, 21:30].values
joint_data_53 = df.iloc[:, 31:40].values
joint_data_54 = df.iloc[:, 41:50].values
joint_data_55 = df.iloc[:, 51:60].values
joint_data_74 = df.iloc[:, 101:110].values

In [ ]:
# for csv have both lower and upper joint data
joint_data_10 = df.iloc[:, 1:10].values
joint_data_11 = df.iloc[:, 11:20].values
joint_data_12 = df.iloc[:, 21:30].values
joint_data_13 = df.iloc[:, 31:40].values
joint_data_14 = df.iloc[:, 41:50].values
joint_data_15 = df.iloc[:, 51:60].values
joint_data_16 = df.iloc[:, 61:70].values
joint_data_30 = df.iloc[:, 71:80].values
joint_data_31 = df.iloc[:, 81:90].values
joint_data_32 = df.iloc[:, 91:100].values
joint_data_33 = df.iloc[:, 101:110].values
joint_data_34 = df.iloc[:, 111:120].values
joint_data_35 = df.iloc[:, 121:130].values
joint_data_36 = df.iloc[:, 131:140].values
joint_data_50 = df.iloc[:, 141:150].values
joint_data_51 = df.iloc[:, 151:160].values
joint_data_52 = df.iloc[:, 161:170].values
joint_data_53 = df.iloc[:, 171:180].values
joint_data_54 = df.iloc[:, 181:190].values
joint_data_55 = df.iloc[:, 191:200].values
joint_data_70 = df.iloc[:, 201:210].values
joint_data_71 = df.iloc[:, 211:220].values
joint_data_72 = df.iloc[:, 221:230].values
joint_data_73 = df.iloc[:, 231:240].values
joint_data_74 = df.iloc[:, 241:250].values
joint_data_75 = df.iloc[:, 251:260].values
joint_data_90 = df.iloc[:, 261:270].values
joint_data_91 = df.iloc[:, 271:280].values
joint_data_92 = df.iloc[:, 281:290].values
joint_data_93 = df.iloc[:, 291:300].values
joint_data_95 = df.iloc[:, 301:310].values

In [ ]:
# all joint pos 先上肢后下肢
fig, axs = plt.subplots(3, 4, figsize=(20, 15), sharex=True)
for i, ax in enumerate(axs.flatten()):
    ax = axs.flatten()[i]
    ax.plot(time_seconds, joint_data[:, i*10+1], label=f'{IP[i]}_pos_{upper_joint_names[i]}')
    ax.plot(time_seconds, joint_data[:, i*10+1+6], label=f'{IP[i]}_targ_pos')
    ax.legend()
fig.suptitle('Upper Joint Position and Target Position')
plt.tight_layout()

In [ ]:
fig, axs = plt.subplots(3, 5, figsize=(20, 15), sharex=True)
for i, ax in enumerate(axs.flatten()):
    ax = axs.flatten()[i]
    ax.plot(time_seconds, joint_data[:, IP_index_lower[IP_lower[i]]], label=f'{IP_lower[i]}_pos_{lower_joint_names[i]}')
    ax.plot(time_seconds, joint_data[:, IP_index_lower[IP_lower[i]]+6], label=f'{IP_lower[i]}_targ_pos')
    ax.legend()
fig.suptitle('Lower Joint Position and Target Position')
plt.tight_layout()

In [ ]:
# all joint data torque
fig, axs = plt.subplots(3, 5, figsize=(20, 15), sharex=True)
for i, ax in enumerate(axs.flatten()):
    ax = axs.flatten()[i]
    ax.plot(time_seconds, joint_data[:, IP_index_lower[IP_lower[i]]+2], label=f'{IP_lower[i]}_cur_{lower_joint_names[i]}')
    ax.plot(time_seconds, joint_data[:, IP_index_lower[IP_lower[i]]+8], label=f'{IP_lower[i]}_targ_cur_{lower_joint_names[i]}')
    # ax.plot(time_seconds, joint_data[:, IP_index_lower[IP_lower[i]]+3], label=f'{IP_lower[i]}_tor_{lower_joint_names[i]}')
    ax.legend()
fig.suptitle('All Joint Torque')
plt.tight_layout()

In [ ]:
# all joint data temperature
fig, axs = plt.subplots(3, 5, figsize=(20, 15), sharex=True)
for i, ax in enumerate(axs.flatten()):
    ax = axs.flatten()[i]
    ax.plot(time_seconds, joint_data[:, IP_index_lower[IP_lower[i]]+4], label=f'{IP_lower[i]}_MOS_{lower_joint_names[i]}')
    ax.plot(time_seconds, joint_data[:, IP_index_lower[IP_lower[i]]+5], label=f'{IP_lower[i]}_Winding_{lower_joint_names[i]}')
    ax.legend()
fig.suptitle('All Joint Temperature')
plt.tight_layout()

In [ ]:
# parse data
pos_54 = joint_data_54[:, 0]
vel_54 = joint_data_54[:, 1]
current_54 = joint_data_54[:, 2]
tor_54 = joint_data_54[:, 3]
mos_temp_54 = joint_data_54[:, 4]
winding_temp_54 = joint_data_54[:, 5]
targ_pos_54 = joint_data_54[:, 6]
targ_vel_54 = joint_data_54[:, 7]
targ_current_54 = joint_data_54[:, 8]

pos_74 = joint_data_74[:, 0]
vel_74 = joint_data_74[:, 1]
current_74 = joint_data_74[:, 2]
tor_74 = joint_data_74[:, 3]
mos_temp_74 = joint_data_74[:, 4]
winding_temp_74 = joint_data_74[:, 5]
targ_pos_74 = joint_data_74[:, 6]
targ_vel_74 = joint_data_74[:, 7]
targ_current_74 = joint_data_74[:, 8]

In [ ]:
# all joint pos 先上肢后下肢
fig, axs = plt.subplots(3, 4, figsize=(20, 15), sharex=True)
for i, ax in enumerate(axs.flatten()):
    ax = axs.flatten()[i]
    ax.plot(time_seconds, joint_data[:, i*10+1], label=f'{IP[i]}_pos_{lower_joint_names[i]}')
    ax.plot(time_seconds, joint_data[:, i*10+1+6], label=f'{IP[i]}_targ_pos')
    ax.legend()
fig.suptitle('All Joint Position and Target Position')
plt.tight_layout()

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(20, 15), sharex=True)
for i in range(6):
    ax = axs[i // 2, i % 2]
    if i < 3:
        ax.plot(time_seconds, joint_data_74[:, i+3], label=f'{labels[i]}_74_targ', linestyle='--')
        ax.plot(time_seconds, joint_data_74[:, i], label=f'{labels[i]}_74')
        ax.plot(time_seconds, joint_data_54[:, i+3], label=f'{labels[i]}_54_targ', linestyle='--')
        ax.plot(time_seconds, joint_data_54[:, i], label=f'{labels[i]}_54')
    else:
        ax.plot(time_seconds, joint_data_74[:, i], label=f'{labels[i]}_74')
        ax.plot(time_seconds, joint_data_54[:, i], label=f'{labels[i]}_54')
    ax.legend()
fig.suptitle('ankle joint data actuator 74 and 54')
plt.tight_layout()